# Motion Planning Portfolio Dashboard

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as img
import numpy as np
import pandas as pd
import seaborn as sns
from pathlib import Path

TRIAL_DIR=Path("../fuklampt/trial-dune")

In [ ]:
df_stats = pd.read_csv(TRIAL_DIR / 'stats.csv.gz')
df_stats['planner_label'] = df_stats.opt_type.str.cat(df_stats.conf_id.astype(str), sep='', join='inner')
df_stats['plan_made'] = ~ df_stats.path_n_points.isna()
df_stats.info()

In [ ]:
summary_cols = ['planner_label','conf_id']+[c for c in df_stats.columns if c.startswith('opt_')]
df_stats.loc[:,summary_cols].drop_duplicates()

## Compare on 'Hard' Queries

In [ ]:
rng = np.random.default_rng(seed=0)
df_stats['qry'] = df_stats.query_sta.str.cat(df_stats.query_end, join='inner', sep=' -> ')
querymeans = df_stats.groupby(['qry'])['time_to_plan'].mean().reset_index(name="mean_plan_time")
print(f"# of queries: {len(querymeans)}")
sample_from = len(querymeans) // 4
sample_size = 12
df_hard_q = querymeans.sort_values('mean_plan_time', ascending=False).iloc[:sample_from]
df_hard_q = df_hard_q.sample(n=sample_size, random_state=rng)
df_hard_q

In [ ]:
df_hard_stats = df_stats.loc[
    df_stats.qry.isin(df_hard_q.qry)&
    (df_stats.plan_made==True)
]

# illustrative pair are sample 4 and 6
to_show = (4,6)

fig,axs = plt.subplots(2, 4, figsize=(12,5), width_ratios=(1.25,1,1,1))
label_order = df_hard_stats.planner_label.unique()
style_opts = {'style' : 'planner_label', 'style_order' : label_order}
hue_opts = {'hue' : 'planner_label', 'hue_order' : label_order}
sns.set_palette('Set2',n_colors=len(label_order))
for qry_id, ((qry,),df) in enumerate(df_hard_stats.groupby(['qry'])):
    if not qry_id in to_show:
        continue
    row = to_show.index(qry_id)

    ax = axs[row][0]
    qry_sta = df.iloc[0].fr_idx
    qry_end = df.iloc[0].to_idx
    thumb = img.imread(TRIAL_DIR / 'thumbnails' / f"qry_{qry_sta:03d}_{qry_end:03d}.png")
    ax.imshow(thumb)
    ax.axis('off')
    ax.set_title(f"Query: {qry_sta:03d} to {qry_end:03d}")

    ax = axs[row][1]
    algo_t_frames = []
    for a in df.planner_label.unique():
        _df = df.loc[df.planner_label == a]
        if len(_df) <= 0:
            print(f"No results for planner {a}, query {qry}")
            continue
        t = pd.DataFrame({
            'time' : [0.0]+sorted(_df['time_to_plan']),
            'n_solved': np.arange(0,len(_df)+1)
        })                     
        t['planner_label'] = a
        algo_t_frames.append(t)
    data = pd.concat(algo_t_frames,ignore_index=True)
    sns.lineplot(ax=ax, data=data, y='n_solved', x='time', **hue_opts)
    ax.set(xlabel="Time (s)" if row!=0 else None, ylabel="# plans solved", xlim=(0,60))
    ax.get_legend().remove()

    ax = axs[row][2]
    data = df.loc[:,['planner_label','time_to_plan']]
    sns.stripplot(ax=ax, data=data, x='time_to_plan', y='planner_label', **hue_opts)
    ax.set(xlabel="Planning time (s)" if row!=0 else None, ylabel=None, xlim=(0,60))
    ax.grid()

    ax = axs[row][3]
    data = df.loc[:,['planner_label','path_exec_time']]
    sns.stripplot(ax=ax, data=data, x='path_exec_time', y='planner_label', **hue_opts)
    ax.set(xlabel="Path exec. time (s)" if row!=0 else None, ylabel=None, xlim=(0,5))
    ax.grid()


fig.tight_layout()
fig.savefig('initial-planner-comparison.pdf', dpi=600)
    

## Performance of Selectors on Test Sets

In [ ]:
def calculate_qcd(df):
    q1 = df.groupby('selector')['time'].quantile(0.25)
    q3 = df.groupby('selector')['time'].quantile(0.75)
    qcd = (q3 - q1) / (q3 + q1)
    return qcd

frames = []

for eval_f in Path(TRIAL_DIR).glob("eval-*csv.gz"):
    name = str(eval_f.name)[len("eval-"):-len(''.join(eval_f.suffixes))]
    df_eval = pd.read_csv(eval_f)
    quants = df_eval.groupby('selector')['time'].quantile((.5,.75,.9,.99,1.))
    means = df_eval.groupby(['split','selector'])['time'].mean().round(2)
    # print(calculate_qcd(df_eval), means, sep="\n")
    
    sel_ord = ['vb_lucky', 'vb_random', 'sb_regress', 'cp_ml_score']
    # fg = sns.catplot(data=df_eval, x='time', y='selector', hue='selector', col='split',
    #                  col_wrap=3, kind='violin', hue_order=sel_ord, sharex=False)
    # fg = sns.catplot(data=df_eval.loc[df_eval.selector.isin(['sb_regress','ml+cp'])], x='time', hue='selector', col='split',
    #                  kind='violin', split=True, sharex=False)

    df_two = df_eval.loc[df_eval.selector.isin(('sb_regress','ml+cp'))]
    pc99 = df_two.groupby(['selector','split'])['time'].quantile(.99).reset_index(name='time')
    pc99['measure'] = 'pc99'
    means = df_two.groupby(['selector','split'])['time'].mean().reset_index(name='time')
    means['measure'] = 'mean'
    both = pd.concat((pc99,means),ignore_index=True)
    both['setup'] = name
    frames.append(both)

print("Displaying the mean and 99th percentile for SB vs ML+CP time")
df_all_setups = pd.concat(frames, ignore_index=True)
data = df_all_setups.pivot_table(index=['measure','split','setup'],values='time',columns=['selector']).reset_index()
fg = sns.relplot(data=data, x='sb_regress', y='ml+cp', col='setup', row='measure', height=2.4, aspect=1, facet_kws=dict(sharex=True))
axis_max = int(100 * (2+ both.time.max() // 100))
fg.set(xlim=(0,axis_max), ylim=(0,axis_max))
(fg.map(plt.axline, xy1=(0,0), xy2=(100,100), color=".5", dashes=(2, 1), zorder=0)
   .map(plt.axline, xy1=(2,1), xy2=(200,100), color=".6", dashes=(3, 1), zorder=0)
   .map(plt.axline, xy1=(5,1), xy2=(500,100), color=".7", dashes=(4, 1), zorder=0)
   .set_axis_labels("SB time (s)", "ML+CP time (s)")
   .set_titles("{col_name} / {row_name}")
   .tight_layout(w_pad=0));


In [ ]:
def renameSelectors(df):
    pretty_names = (
        ('vb_lucky',"VB Lucky"), 
        ('vb_random' , "VB Random"),
        ('sb_regress' , "SB + ML Regression"),
        ('ml+cp' , "PAMPR (ML+CP)")
    )
    ranked_selectors = [t[0] for t in pretty_names]
    ranks_dict = { n:str(r) for r,n in enumerate(ranked_selectors)}
    df['selector_rank'] = df.selector.str.replace(ranks_dict).astype('str')
    df['selector'] = df.selector.str.replace(dict(pretty_names))
    return df

In [ ]:
for eval_f in sorted(TRIAL_DIR.glob('eval-*.csv.gz')):
    df_eval = renameSelectors(pd.read_csv(eval_f))
    name = str(eval_f.name)[len("eval-"):-len(''.join(eval_f.suffixes))]
    data = df_eval.loc[df_eval.selector!='VB Lucky']
    ho=("VB Random","SB + ML Regression","PAMPR (ML+CP)")
    vb_lucky_vals = df_eval.loc[df_eval.selector=='VB Lucky'].groupby(['split'])['time'].median()
    fg = sns.displot(data, x='time', col='split', col_wrap=5, hue='selector', kind='kde',
                     height=2, aspect=1.25, hue_order=ho, facet_kws=dict(sharex=False, sharey=False, legend_out=False))
    sns.move_legend(fg,"lower center", bbox_to_anchor=(0.5, 0.01), frameon=True, ncols=3, title="Selector")
    for col, ax in enumerate(fg.axes):
        vb_lucky_val = vb_lucky_vals.loc[col]
        ax.axvline(vb_lucky_val, c=".5", dashes=(2,1), zorder=0)
        ax.set_yticklabels([])
        ax.set_yticks([])
    fg.set_axis_labels("Time (s)", "Freq. Density")
    fg.set_titles("Test set {col_name}")
    fg.figure.tight_layout(rect=(0, 0.15, 1, 1))
    fg.figure.savefig(f"test-set-distrib-{name}.pdf", dpi=600)


## Summary Stats for Paper

In [ ]:
frames = []
for eval_f in sorted(TRIAL_DIR.glob('eval-*.csv.gz')):
    name = str(eval_f.name)[len("eval-"):-len(''.join(eval_f.suffixes))]
    if name.startswith('score,'):
        continue
    df_eval = pd.read_csv(eval_f)
    df_eval['setup'] = name
    frames.append(df_eval)
df_all = renameSelectors(pd.concat(frames, ignore_index=True))
time_aggs = [
    ('mean','mean'),
    ('95 percentile',lambda x:x.quantile(.95)),
    ('max','max'),
]
time_summary = (
    df_all.groupby(['setup','selector_rank','selector'])['time']
          .agg(time_aggs)
          .round(2)
          .reset_index()
          .drop(columns=['selector_rank'])
)
print(time_summary,time_summary.style.format(precision=2).hide(axis=0).to_latex(),sep="\n")

tout_summary = (
    df_all.groupby(['setup','selector_rank','selector'])['n_timeouts']
          .agg(['mean','max'])
          .round(2)
          .reset_index()
          .drop(columns=['selector_rank'])
)
print(tout_summary,tout_summary.style.format(precision=2).hide(axis=0).to_latex(),sep="\n")

